In [132]:
import numpy as np
import pandas as pd
from sklearn.linear_model import TweedieRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings('ignore')

In [136]:
# Variables
source_path = r"C:\Users\ASUS\Desktop\AUA books\Y4S2\Capstone\code\database\yerevan_combined_clean/historical_sold.csv"
output_path = r"C:\Users\ASUS\Desktop\AUA books\Y4S2\Capstone\code\database\cross-validation-glr.csv"

In [134]:
# Load the data
data = pd.read_csv(source_path)
data = data.dropna(subset=['square_meters'])

In [135]:
# Get the list of unique neighborhoods
neighborhoods = data['neighborhood'].unique()

# Set the significance level for the paired t-test
significance_level = 0.05

# Initialize a list to store the results for each neighborhood
results = []

# Loop through each neighborhood and calculate the training and testing errors
for neighborhood in neighborhoods:
    # Extract data for the current neighborhood
    neighborhood_data = data[data['neighborhood'] == neighborhood]
    
    # Split the data into independent (X) and dependent (y) variables
    X = neighborhood_data[['square_meters']]
    y = neighborhood_data['price']
    
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize your GLR model with the appropriate distribution and link function
    glr_model = TweedieRegressor(power=0, link='identity')
    
    # Fit the model on the training data
    glr_model.fit(X_train, y_train)
    
    # Evaluate the model on the training set
    y_train_pred = glr_model.predict(X_train)
    train_error = mean_squared_error(y_train, y_train_pred)
    
    # Evaluate the model on the testing set
    y_test_pred = glr_model.predict(X_test)
    test_error = mean_squared_error(y_test, y_test_pred)
    
    # Calculate the mean price of the training data
    mean_price = y_train.mean()
    
    # Predict the mean price for the testing set
    y_baseline = [mean_price] * len(y_test)
    
    # Calculate the testing error for the baseline method
    baseline_error = mean_squared_error(y_test, y_baseline)
    
    # Concatenate the training and testing sets before performing the paired t-test
    y_concat = np.concatenate([y_train, y_test])
    y_pred_concat = np.concatenate([y_train_pred, y_test_pred])
    
    # Perform the paired t-test between the training and testing errors
    t_stat, p_val = ttest_rel(y_concat, y_pred_concat)
    
    # Store the results for the current neighborhood in a dictionary
    result = {'Neighborhood': neighborhood,
              'Training error': train_error,
              'Testing error (GLR model)': test_error,
              'Testing error (baseline method)': baseline_error,
              'Paired t-test p-value': p_val}
    
    # Compare the testing error of the GLR model and the baseline method
    if test_error < baseline_error:
        improvement = (baseline_error - test_error) / baseline_error * 100
        result['Improvement over baseline (%)'] = improvement
    elif test_error > baseline_error:
        difference = (test_error - baseline_error) / baseline_error * 100
        result['Improvement over GLR model (%)'] = difference
    else:
        result['Improvement over baseline (%)'] = 0.0
        result['Improvement over GLR model (%)'] = 0.0
    
    # Check if the p-value is less than the significance level
    if p_val < significance_level:
        result['Warning'] = 'GLR model may be overfitting the training data!'
    
    # Append the result to
    results.append(result)

# Convert the list of dictionaries to a DataFrame
results_df = pd.DataFrame(results)

# Save the DataFrame to a CSV file
results_df.to_csv(output_path, index=False)